## 0. Google Colab Setup

Mount Google Drive to access the project directory. Run this cell first in every session to establish the working path.

In [12]:
!pip install -r /content/drive/MyDrive/multimodal-causal-ablation/requirements.txt transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 798.0 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 110.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 69.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
# Mount to the standard base directory
drive.mount('/content/drive')

# Now you can define your project path and use it
project_path = '/content/drive/MyDrive/multimodal-causal-ablation'
import os
if os.path.exists(project_path):
    print(f'Successfully accessed: {project_path}')
else:
    print(f'Drive mounted, but folder not found: {project_path}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully accessed: /content/drive/MyDrive/multimodal-causal-ablation


# Phase A — Dominant Modality Verification

Verify Audio as the dominant modality via aggregated DeepSHAP attribution, following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Formally confirm which modality contributes the highest aggregated DeepSHAP attribution score for both the base and fine-tuned models. This is the prerequisite gate before any neuron-level probing or ablation work in Phases B–D.

## 1. Environment & Imports

Set up the deterministic seed (`seed=0`, matching upstream checkpoint convention) and import all required libraries. The seed utility from `src/utils.py` pins `torch`, `numpy`, `random`, and `cudnn` for full reproducibility across ephemeral Colab runtimes.

**Expected output:** Confirmation of project path, checkpoints directory, and results directory.

In [2]:
import sys
import os
import pickle

import numpy as np
import pandas as pd

# Add project src to path for utility imports
sys.path.insert(0, os.path.join(project_path, 'src'))
from utils import set_deterministic_seed

# Pin all randomness sources (seed=0 matches upstream checkpoint convention)
set_deterministic_seed(seed=0)

# Define paths
checkpoints_dir = os.path.join(project_path, 'checkpoints')
results_dir = os.path.join(project_path, 'results')
os.makedirs(results_dir, exist_ok=True)

print(f'Project path:  {project_path}')
print(f'Checkpoints:   {checkpoints_dir}')
print(f'Results:       {results_dir}')

Project path:  /content/drive/MyDrive/multimodal-causal-ablation
Checkpoints:   /content/drive/MyDrive/multimodal-causal-ablation/checkpoints
Results:       /content/drive/MyDrive/multimodal-causal-ablation/results


## 2. Load Pre-computed DeepSHAP Attributions

Load the pre-computed DeepSHAP attribution pickles for both the base and fine-tuned models. Each pickle contains a dict with:
- `'SHAP_value'`: list of 6 numpy arrays (one per emotion class), each shaped `(n_samples, 1152)`
- `'test_feature'`: corresponding test input features

The 1152-dimensional feature vector is a concatenation of three modality representations:
- **Text** (ALBERT): dimensions 0–1023 (1024-d)
- **Video** (visual): dimensions 1024–1087 (64-d)
- **Audio** (acoustic): dimensions 1088–1151 (64-d)

**Expected output:** Structure summary showing keys, number of classes, and per-class array shapes for both models.

In [3]:
# Load pre-computed DeepSHAP attributions for both models
with open(os.path.join(checkpoints_dir, 'base_shap.pkl'), 'rb') as f:
    base_shap_data = pickle.load(f)

with open(os.path.join(checkpoints_dir, 'finetuned_shap.pkl'), 'rb') as f:
    finetuned_shap_data = pickle.load(f)

# Inspect structure
print('=== Base Model SHAP ===')
print(f'Keys: {list(base_shap_data.keys())}')
print(f'Number of classes: {len(base_shap_data["SHAP_value"])}')
for i, sv in enumerate(base_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

print()
print('=== Fine-tuned Model SHAP ===')
print(f'Keys: {list(finetuned_shap_data.keys())}')
print(f'Number of classes: {len(finetuned_shap_data["SHAP_value"])}')
for i, sv in enumerate(finetuned_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

=== Base Model SHAP ===
Keys: ['SHAP_value', 'test_feature']
Number of classes: 6
  Class 0: shape = (144, 1152)
  Class 1: shape = (144, 1152)
  Class 2: shape = (144, 1152)
  Class 3: shape = (144, 1152)
  Class 4: shape = (144, 1152)
  Class 5: shape = (144, 1152)

=== Fine-tuned Model SHAP ===
Keys: ['SHAP_value', 'test_feature']
Number of classes: 6
  Class 0: shape = (144, 1152)
  Class 1: shape = (144, 1152)
  Class 2: shape = (144, 1152)
  Class 3: shape = (144, 1152)
  Class 4: shape = (144, 1152)
  Class 5: shape = (144, 1152)


## 3. Compute Aggregated SHAP Attribution per Modality

Raw 1152-d SHAP values create a **dimensionality illusion**: Text has 16× more features than Audio or Video, so naive per-feature comparisons inflate Text's apparent contribution.

Per ADR 0001, I resolve this by computing **aggregated** SHAP attribution:
1. For each emotion class, take the absolute value of all SHAP values.
2. Sum |SHAP| within each modality for every sample — this collapses each modality's contribution to a single scalar per sample.
3. Average across samples to get one attribution score per modality per class.
4. Average across classes to get the overall modality attribution.

This makes the comparison fair regardless of how many raw features each modality contributes.

**Expected output:** Per-class attribution percentages for Text, Video, and Audio in both models.

In [4]:
# Modality feature ranges in the 1152-d concatenated feature vector
MODALITY_RANGES = {
    'Text':  (0, 1024),     # ALBERT embeddings (1024-d)
    'Video': (1024, 1088),  # Visual features (64-d)
    'Audio': (1088, 1152),  # Acoustic features (64-d)
}

EMOTION_CLASSES = [
    'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise'
]


def compute_aggregated_shap(shap_data, model_name):
    """Compute aggregated SHAP attribution per modality.

    For each class, sum |SHAP values| within each modality per sample,
    then average across samples. This is the aggregation method from
    ADR 0001 to resolve the dimensionality illusion between Text
    (1024-d) and Audio (64-d).
    """
    shap_values = shap_data['SHAP_value']
    n_classes = len(shap_values)

    rows = []
    for class_idx in range(n_classes):
        sv = shap_values[class_idx]  # (n_samples, 1152)

        row = {'model': model_name, 'class': EMOTION_CLASSES[class_idx]}
        for mod_name, (start, end) in MODALITY_RANGES.items():
            # Sum |SHAP| within modality per sample, then mean across samples
            per_sample = np.sum(np.abs(sv[:, start:end]), axis=1)
            row[f'{mod_name}_attribution'] = np.mean(per_sample)
            row[f'{mod_name}_std'] = np.std(per_sample)

        # Percentages for readability
        total = sum(row[f'{m}_attribution'] for m in MODALITY_RANGES)
        for mod_name in MODALITY_RANGES:
            row[f'{mod_name}_pct'] = (
                row[f'{mod_name}_attribution'] / total * 100
            )

        rows.append(row)

    return pd.DataFrame(rows)


# Compute for both models
base_attr_df = compute_aggregated_shap(base_shap_data, 'base')
finetuned_attr_df = compute_aggregated_shap(finetuned_shap_data, 'finetuned')

# Combine results
attribution_df = pd.concat(
    [base_attr_df, finetuned_attr_df], ignore_index=True
)

# Display per-class results
print('=== Per-Class Aggregated SHAP Attribution (%) ===')
print()
display_cols = ['model', 'class', 'Text_pct', 'Video_pct', 'Audio_pct']
print(
    attribution_df[display_cols].to_string(
        index=False, float_format='%.2f'
    )
)

=== Per-Class Aggregated SHAP Attribution (%) ===

    model     class  Text_pct  Video_pct  Audio_pct
     base     anger     71.20       2.65      26.15
     base   disgust     74.49       3.50      22.01
     base      fear     75.74       2.89      21.37
     base happiness     70.90       3.35      25.76
     base   sadness     72.32       2.82      24.85
     base  surprise     73.73       2.60      23.67
finetuned     anger     66.99       1.39      31.62
finetuned   disgust     68.20       2.00      29.80
finetuned      fear     70.55       1.27      28.19
finetuned happiness     68.54       2.04      29.41
finetuned   sadness     67.43       1.67      30.90
finetuned  surprise     70.08       1.10      28.82


In [5]:
def compute_verdict(attr_df, model_name):
    """Apply ADR 0001 dominant modality decision rule.

    Returns a dict with the verdict and supporting evidence.
    """
    model_df = attr_df[attr_df['model'] == model_name]

    # Overall attribution: mean across classes
    summary = {}
    for mod_name in MODALITY_RANGES:
        summary[mod_name] = model_df[f'{mod_name}_attribution'].mean()

    total = sum(summary.values())
    pct = {k: v / total * 100 for k, v in summary.items()}

    # Rank by attribution
    ranked = sorted(pct.items(), key=lambda x: x[1], reverse=True)

    sep = '=' * 55
    print(sep)
    print(f'  {model_name.upper()} MODEL — Aggregated SHAP Attribution')
    print(sep)
    for mod, p in ranked:
        print(f'  {mod:8s}: {p:6.2f}%  (raw mean: {summary[mod]:.6f})')

    top_mod, top_pct = ranked[0]
    second_mod, second_pct = ranked[1]
    gap = top_pct - second_pct

    # ADR 0001 decision rule
    if gap <= 5.0:
        dominant = 'Audio'
        reason = (
            f'Top two modalities ({top_mod}: {top_pct:.2f}%, '
            f'{second_mod}: {second_pct:.2f}%) are within 5% parity '
            f'(gap = {gap:.2f}%). Per ADR 0001, targeting Audio (64-d) '
            f'for superior neuron-to-class ratio.'
        )
    else:
        dominant = top_mod
        reason = (
            f'{top_mod} leads with {top_pct:.2f}% vs '
            f'{second_mod} at {second_pct:.2f}% '
            f'(gap = {gap:.2f}% > 5% threshold).'
        )

    print()
    print(f'  VERDICT: Dominant Modality = {dominant}')
    print(f'  Reason:  {reason}')

    return {
        'model': model_name,
        'dominant_modality': dominant,
        'Text_pct': round(pct['Text'], 4),
        'Video_pct': round(pct['Video'], 4),
        'Audio_pct': round(pct['Audio'], 4),
        'top_modality': top_mod,
        'second_modality': second_mod,
        'gap_pct': round(gap, 4),
        'parity_rule_applied': gap <= 5.0,
        'reason': reason,
    }


# Apply verdict to both models
base_verdict = compute_verdict(attribution_df, 'base')
print()
finetuned_verdict = compute_verdict(attribution_df, 'finetuned')

# --- Save results ---
verdict_df = pd.DataFrame([base_verdict, finetuned_verdict])
verdict_path = os.path.join(
    results_dir, 'phase_a_dominant_modality_verdict.csv'
)
verdict_df.to_csv(verdict_path, index=False)

detail_path = os.path.join(
    results_dir, 'phase_a_shap_attribution_by_class.csv'
)
attribution_df.to_csv(detail_path, index=False)

print()
print(f'Results saved:')
print(f'  Verdict:  {verdict_path}')
print(f'  Details:  {detail_path}')

# --- Summary ---
sep = '=' * 55
print()
print(sep)
print('  PHASE A SUMMARY')
print(sep)
bm = base_verdict['dominant_modality']
fm = finetuned_verdict['dominant_modality']
print(f'  Base model dominant modality:       {bm}')
print(f'  Fine-tuned model dominant modality:  {fm}')

if base_verdict['dominant_modality'] == 'Audio':
    print()
    print('  ✓ Audio confirmed as dominant modality.')
    print('    Proceed to Phase B: Probe Signal Validation '
          'on Audio FFN activations.')
else:
    dm = base_verdict['dominant_modality']
    print()
    print(f'  ✗ Audio NOT confirmed. Dominant modality = {dm}.')
    print('    Review methodology — experiment targets '
          'the dominant modality.')

  BASE MODEL — Aggregated SHAP Attribution
  Text    :  73.17%  (raw mean: 0.258486)
  Audio   :  23.84%  (raw mean: 0.084228)
  Video   :   2.99%  (raw mean: 0.010561)

  VERDICT: Dominant Modality = Text
  Reason:  Text leads with 73.17% vs Audio at 23.84% (gap = 49.33% > 5% threshold).

  FINETUNED MODEL — Aggregated SHAP Attribution
  Text    :  68.67%  (raw mean: 0.265253)
  Audio   :  29.77%  (raw mean: 0.114995)
  Video   :   1.56%  (raw mean: 0.006042)

  VERDICT: Dominant Modality = Text
  Reason:  Text leads with 68.67% vs Audio at 29.77% (gap = 38.90% > 5% threshold).

Results saved:
  Verdict:  /content/drive/MyDrive/multimodal-causal-ablation/results/phase_a_dominant_modality_verdict.csv
  Details:  /content/drive/MyDrive/multimodal-causal-ablation/results/phase_a_shap_attribution_by_class.csv

  PHASE A SUMMARY
  Base model dominant modality:       Text
  Fine-tuned model dominant modality:  Text

  ✗ Audio NOT confirmed. Dominant modality = Text.
    Review methodology —

# Phase B — Probe Signal Validation

Validate the target modality's signal via L1-logistic regression, following the methodology locked in [ADR         
0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Ensure the chosen modality branch (Text, 1024-d) retains sufficient class-discriminative information     
before we perform causal ablations. If the signal is too weak, we trigger a layer fallback.

## 4. Resume Gate: Load Cached Activations
Loads cached 1024-d Text branch activations to bypass expensive PyTorch forward passes.

**Expected output:** Confirmation of loaded tensor shapes.

In [6]:
import os                                                                                                          
import pickle                                                                                                      
import numpy as np                                                                                                 
import torch                                                                                                       
                                                                                                                    
# Use absolute paths for Colab                                                                                     
checkpoints_dir = os.path.join(project_path, 'checkpoints')                                                        
activations_dir = os.path.join(checkpoints_dir, 'activations')                                                     
data_dir = os.path.join(project_path, 'Model/Dig-Data_Model-Main/data')                                            
                                                                                                                    
print("Loading SHAP data to extract Tier 2 representations...")                                                    
with open(os.path.join(checkpoints_dir, 'base_shap.pkl'), 'rb') as f:                                              
    base_shap_data = pickle.load(f)                                                                                
                                                                                                                    
with open(os.path.join(checkpoints_dir, 'finetuned_shap.pkl'), 'rb') as f:                                         
    ft_shap_data = pickle.load(f)                                                                                  
                                                                                                                    
# 1. Extract the first 1024 dimensions (ALBERT CLS outputs)                                                        
base_test_feats = base_shap_data['test_feature']                                                                   
ft_test_feats = ft_shap_data['test_feature']                                                                       
                                                                                                                    
if torch.is_tensor(base_test_feats): base_test_feats = base_test_feats.cpu().numpy()                               
if torch.is_tensor(ft_test_feats): ft_test_feats = ft_test_feats.cpu().numpy()                                     
                                                                                                                    
base_acts_tier2 = base_test_feats[:, 0:1024]                                                                       
ft_acts_tier2 = ft_test_feats[:, 0:1024]                                                                           
                                                                                                                    
# Cache them specifically as tier 2
np.save(os.path.join(activations_dir, 'base_acts_tier2.npy'), base_acts_tier2)
np.save(os.path.join(activations_dir, 'ft_acts_tier2.npy'), ft_acts_tier2)

# 2. Extract True Labels directly from RML metadata
emoDict = {'ang': 0, 'dis': 1, 'fea': 2, 'hap': 3, 'sad': 4, 'sur': 5}
split_file = os.path.join(data_dir, 'data_split', 'all_single_label_six_category', 'with_valid', 'Final_test_split_six_categories_RML.txt')
meta_file = os.path.join(data_dir, 'RML_RAW_PROCESSED_Face', 'meta.pkl')

with open(split_file, 'r') as f:
    uttr_ids = f.read().splitlines()

with open(meta_file, 'rb') as f:
    meta = pickle.load(f)

# Map utterance ID -> string label -> integer label
labels_tier2 = np.array([emoDict[meta[uid]['label']] for uid in uttr_ids])

print(f"Successfully extracted Tier 2 (Encoder CLS) 1024-d representations and labels.")
print(f"  Base Tier 2 shape: {base_acts_tier2.shape}")
print(f"  FT Tier 2 shape:   {ft_acts_tier2.shape}")
print(f"  Labels shape:      {labels_tier2.shape}")

Loading SHAP data to extract Tier 2 representations...
Successfully extracted Tier 2 (Encoder CLS) 1024-d representations and labels.
  Base Tier 2 shape: (144, 1024)
  FT Tier 2 shape:   (144, 1024)
  Labels shape:      (144,)


## 5. L1-Logistic Probe Validation (Tier 1)

*Note on Data Integrity:* Initial tests on a pre-computed `(288, 512)` cache showed zero signal. A forensic audit revealed this cache belonged to a different dataset (likely IEMOCAP video features). Furthermore, a review of `src/models/e2e.py` confirms that the Text branch lacks an intermediate Feed-Forward Network. The ALBERT Encoder CLS token (1024-d) routes directly to the classifier. Therefore, this 1024-d vector represents our true **Tier 1** representation.

Below, we extract the correct 1024-d text features and the corresponding 144 RML true labels to perform the Tier 1 validation.

**Expected output:** Per-class AUC scores, Mean AUC, and a PASS/WARNING fallback decision for both the base and fine-tuned models.

In [7]:
import os
import pickle
import numpy as np
import torch

checkpoints_dir = os.path.join(project_path, 'checkpoints')
activations_dir = os.path.join(checkpoints_dir, 'activations')
data_dir = os.path.join(project_path, 'Model/Dig-Data_Model-Main/data')

print("Extracting 1024-d Tier 1 Activations from SHAP cache...")
with open(os.path.join(checkpoints_dir, 'base_shap.pkl'), 'rb') as f:
    base_shap_data = pickle.load(f)
with open(os.path.join(checkpoints_dir, 'finetuned_shap.pkl'), 'rb') as f:
    ft_shap_data = pickle.load(f)

base_acts_tier1 = base_shap_data['test_feature'][:, 0:1024]
ft_acts_tier1 = ft_shap_data['test_feature'][:, 0:1024]
if torch.is_tensor(base_acts_tier1): base_acts_tier1 = base_acts_tier1.cpu().numpy()
if torch.is_tensor(ft_acts_tier1): ft_acts_tier1 = ft_acts_tier1.cpu().numpy()

# Save pristine copies
np.save(os.path.join(activations_dir, 'base_acts_tier1.npy'), base_acts_tier1)
np.save(os.path.join(activations_dir, 'ft_acts_tier1.npy'), ft_acts_tier1)

print(f"Base Tier 1 shape: {base_acts_tier1.shape}")
print(f"FT Tier 1 shape:   {ft_acts_tier1.shape}")

Extracting 1024-d Tier 1 Activations from SHAP cache...
Base Tier 1 shape: (144, 1024)
FT Tier 1 shape:   (144, 1024)


### 5.1 Aligning Target Labels
The previously cached labels were tied to the corrupted 288-sample dataset. Here, we cleanly extract the true 144 target labels for the RML dataset by matching the test split utterance IDs directly against the dataset's `meta.pkl`.

In [8]:
print("Extracting True RML Labels from metadata...")
emoDict = {'ang': 0, 'dis': 1, 'fea': 2, 'hap': 3, 'sad': 4, 'sur': 5}
split_file = os.path.join(data_dir, 'data_split', 'all_single_label_six_category', 'with_valid', 'Final_test_split_six_categories_RML.txt')
meta_file = os.path.join(data_dir, 'RML_RAW_PROCESSED_Face', 'meta.pkl')

with open(split_file, 'r') as f:
    uttr_ids = f.read().splitlines()
with open(meta_file, 'rb') as f:
    meta = pickle.load(f)

labels_tier1 = np.array([emoDict[meta[uid]['label']] for uid in uttr_ids])
print(f"Labels shape:      {labels_tier1.shape}")

Extracting True RML Labels from metadata...
Labels shape:      (144,)


### 5.2 Out-of-Fold Evaluation
We define and run our out-of-fold `StratifiedKFold` logistic regression pipeline on the clean Tier 1 representations. If the Mean AUC exceeds 0.65 and at least 4 out of 6 classes exceed 0.55, the signal validation formally passes.

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

EMOTION_CLASSES = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']

def validate_probe_signal(acts, labels, model_name):
    print(f"=== {model_name.upper()} MODEL PROBE VALIDATION (Tier 1) ===")
    n_classes = len(np.unique(labels))
    scaler = StandardScaler()
    acts_scaled = scaler.fit_transform(acts)
    aucs = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    
    for c in range(n_classes):
        y_binary = (labels == c).astype(int)
        probe = LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced', random_state=0, max_iter=1000)
        probs = cross_val_predict(probe, acts_scaled, y_binary, cv=cv, method='predict_proba')[:, 1]
        auc = roc_auc_score(y_binary, probs)
        aucs.append(auc)
        print(f"  Class {EMOTION_CLASSES[c]:<10} AUC: {auc:.4f}")
        
    mean_auc = np.mean(aucs)
    poor_classes = sum(1 for a in aucs if a < 0.55)
    print("-" * 45)
    print(f"  Mean AUC: {mean_auc:.4f} (Threshold: 0.65)")
    print(f"  Classes < 0.55 AUC: {poor_classes} (Threshold: <= 2)")
    
    if mean_auc < 0.65 or poor_classes > 2:
        print(f"  [WARNING] Tier 1 Signal Validation FAILED for {model_name}.")
        return False
    print(f"  [PASS] Tier 1 Signal Validation SUCCEEDED for {model_name}.")
    return True

print("Validating True Tier 1 Signal...\n")
base_valid = validate_probe_signal(base_acts_tier1, labels_tier1, 'base')
print()
ft_valid = validate_probe_signal(ft_acts_tier1, labels_tier1, 'finetuned')

if base_valid and ft_valid:
    print("\n✓ Both models passed Tier 1 validation. Proceeding to Phase C (Causal Ablation).")

Validating True Tier 1 Signal...

=== BASE MODEL PROBE VALIDATION (Tier 1) ===
  Class anger      AUC: 0.7368
  Class disgust    AUC: 0.6944
  Class fear       AUC: 0.6870
  Class happiness  AUC: 0.7450
  Class sadness    AUC: 0.7200
  Class surprise   AUC: 0.8314
---------------------------------------------
  Mean AUC: 0.7358 (Threshold: 0.65)
  Classes < 0.55 AUC: 0 (Threshold: <= 2)
  [PASS] Tier 1 Signal Validation SUCCEEDED for base.

=== FINETUNED MODEL PROBE VALIDATION (Tier 1) ===
  Class anger      AUC: 0.5342
  Class disgust    AUC: 0.6646
  Class fear       AUC: 0.6351
  Class happiness  AUC: 0.7182
  Class sadness    AUC: 0.7085
  Class surprise   AUC: 0.8023
---------------------------------------------
  Mean AUC: 0.6771 (Threshold: 0.65)
  Classes < 0.55 AUC: 1 (Threshold: <= 2)
  [PASS] Tier 1 Signal Validation SUCCEEDED for finetuned.

✓ Both models passed Tier 1 validation. Proceeding to Phase C (Causal Ablation).


# Phase C — Causal Ablation

Compute class-selectivity ratios and perform mean-ablation sweeps, following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Identify the most causally active neurons for each emotion class in the 1024-d Text Tier 1 representation, and evaluate the classification accuracy drop when these specific neurons are ablated. A feature set is causally class-selective if the target class accuracy drop is $\ge 2.5\times$ the mean absolute non-target class drop.

## 6. Compute Class-Selectivity Ratios

To find the top $k$ neurons to ablate, we compute the selectivity ratio $\frac{\mu(n|c)}{\mu(n|\neg c)}$ for every neuron $n$ in the 1024-d representation across all emotion classes.

**Expected output:** A list of the top 5 highly selective neuron indices for each class in both the base and fine-tuned models.

In [10]:
import numpy as np

# Define classes and k-values
EMOTION_CLASSES = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']
K_VALUES = [1, 3, 5, 10]

print("=== Phase C: Computing Class-Selectivity Ratios ===")

def compute_top_neurons(acts, labels):
    """Computes the selectivity ratio and returns top neurons for each class."""
    n_classes = len(np.unique(labels))
    top_neurons = {}
    
    for c in range(n_classes):
        target_acts = acts[labels == c]
        off_target_acts = acts[labels != c]
        
        # Calculate mean activations per neuron
        mean_target = np.mean(target_acts, axis=0)
        mean_off_target = np.mean(off_target_acts, axis=0)
        
        # Avoid division by zero
        eps = 1e-8
        selectivity_ratio = (mean_target + eps) / (mean_off_target + eps)
        
        # Sort and get top 10 neurons
        top_10 = np.argsort(selectivity_ratio)[-10:][::-1]
        top_neurons[c] = top_10
        
    return top_neurons

# Reusing base_acts_tier1, ft_acts_tier1, labels_tier1 from Phase B memory
base_top_neurons = compute_top_neurons(base_acts_tier1, labels_tier1)
ft_top_neurons = compute_top_neurons(ft_acts_tier1, labels_tier1)

print("\nBase Model - Top 5 Selective Neurons per Class:")
for c, idxs in base_top_neurons.items():
    print(f"  {EMOTION_CLASSES[c]:<10}: {idxs[:5]}")

print("\nFine-tuned Model - Top 5 Selective Neurons per Class:")
for c, idxs in ft_top_neurons.items():
    print(f"  {EMOTION_CLASSES[c]:<10}: {idxs[:5]}")

=== Phase C: Computing Class-Selectivity Ratios ===

Base Model - Top 5 Selective Neurons per Class:
  anger     : [ 25 574  48 136 650]
  disgust   : [283 517 412 853 969]
  fear      : [541 849 156 674 858]
  happiness : [335 411 485 173 612]
  sadness   : [773 776 620   6 880]
  surprise  : [447 418 669 626 249]

Fine-tuned Model - Top 5 Selective Neurons per Class:
  anger     : [956 885 459 986 577]
  disgust   : [ 22 849 478  26  95]
  fear      : [448  52 834 955 849]
  happiness : [ 834  747  866 1015  412]
  sadness   : [510 715 525 335  90]
  surprise  : [658 437 850 735 776]


## 7. Mean-Ablation Proxy Inference & Model Loading

Because causal ablation must be measured via accuracy drops, we must evaluate the actual PyTorch `MME2E` models. However, instead of reloading the raw dataset (images, audio, text) and running the heavy encoders, we can leverage the exact 1152-d `test_feature` representations we already cached in Phase A. 

The `MME2E` architecture allows us to cleanly split this 1152-d vector back into `Text (1024-d)`, `Video (64-d)`, and `Audio (64-d)`, and pass them directly into the pre-trained classification heads (`t_out`, `v_out`, `a_out`, and `weighted_fusion`). This "proxy inference" mathematically perfectly simulates the forward pass while allowing us to seamlessly clamp the Text neurons.

**Expected output:** Loading of the base and fine-tuned `MME2E` PyTorch checkpoints, and definition of the `ablate_and_evaluate` proxy inference function.

In [13]:
import sys
import os
import torch
import torch.nn as nn
import numpy as np

# Ensure MME2E is in path
sys.path.insert(0, os.path.join(project_path, 'Model/Dig-Data_Model-Main'))
from src.models.e2e import MME2E

# Setup mocked args for model instantiation (matches Phase A setup)
args = {
    'num_emotions': 6,
    'modalities': 'tav',
    'feature_dim': 256,
    'trans_nlayers': 4,
    'trans_nheads': 4,
    'trans_dim': 64,
    'text_model_size': 'large',
}
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("Loading PyTorch MME2E models for ablation...")
base_model = MME2E(args=args, device=device).to(device)
base_model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'base_model.pt'), map_location=device), strict=False)
base_model.eval()

ft_model = MME2E(args=args, device=device).to(device)
ft_model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'finetuned_model.pt'), map_location=device), strict=False)
ft_model.eval()
print("Models loaded successfully.")

# Extract the full 1152-d test features
print("Loading 1152-d full representations...")
base_1152 = base_shap_data['test_feature']
ft_1152 = ft_shap_data['test_feature']
if torch.is_tensor(base_1152): base_1152 = base_1152.cpu().numpy()
if torch.is_tensor(ft_1152): ft_1152 = ft_1152.cpu().numpy()

def ablate_and_evaluate(model, acts_1152, labels, top_k_neurons, dataset_mean_acts):
    """
    Proxy inference ablation: splits the 1152-d vector, clamps top-k Text neurons, 
    passes through final classification heads, and returns per-class accuracy.
    """
    # Clone to avoid mutating the original cache
    text_cls = acts_1152[:, 0:1024].copy()
    faces = acts_1152[:, 1024:1088]
    specs = acts_1152[:, 1088:1152]
    
    # Ablation clamp on the 1024-d text modality
    for n in top_k_neurons:
        text_cls[:, n] = dataset_mean_acts[n]
        
    with torch.no_grad():
        t_t = torch.tensor(text_cls, dtype=torch.float32).to(device)
        f_t = torch.tensor(faces, dtype=torch.float32).to(device)
        s_t = torch.tensor(specs, dtype=torch.float32).to(device)
        
        # Pass directly into the classification heads
        t_logits = model.t_out(t_t)
        v_logits = model.v_out(f_t)
        a_logits = model.a_out(s_t)
        
        all_logits = torch.stack([t_logits, v_logits, a_logits], dim=-1)
        final_logits = model.weighted_fusion(all_logits).squeeze(-1)
        preds = torch.argmax(final_logits, dim=1).cpu().numpy()
    
    # Calculate per-class accuracy
    accs = []
    for c in range(6):
        mask = (labels == c)
        acc = np.mean(preds[mask] == labels[mask]) if np.sum(mask) > 0 else 0.0
        accs.append(acc)
        
    return np.array(accs)

print("Ablation proxy inference scaffold ready.")

Loading PyTorch MME2E models for ablation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 71.5MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/drive/MyDrive/multimodal-causal-ablation/Model/Dig-Data_Model-Main/src/models/transformer_encoder.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully.
Loading 1152-d full representations...
Ablation proxy inference scaffold ready.


## 8. Execute Causal Selectivity Sweep & Evaluation

We now compute the baseline accuracy and evaluate the accuracy drop for ablating the top $k \in \{1, 3, 5, 10\}$ neurons.

According to ADR 0001, a feature set is causally class-selective if:
`Target Class Accuracy Drop` $\ge 2.5\times$ `Mean Absolute Non-Target Class Drop`.

**Expected output:** Tables reporting the baseline accuracy, the ablation accuracy drops for $k=5$ (primary reporting target), and the final causal selectivity pass/fail per class.

In [16]:
import pandas as pd
import json

print("=== Phase C: Causal Ablation Sweep ===")

def run_sweep(model, acts_1152, acts_tier1, labels, top_neurons, model_name):
    # Compute dataset-mean activations for clamping (using 1024-d Tier 1)
    dataset_mean_acts = np.mean(acts_tier1, axis=0)
    
    # Compute baseline accuracy (k=0 ablation)
    baseline_accs = ablate_and_evaluate(model, acts_1152, labels, [], dataset_mean_acts)
    
    results = []
    # Evaluate for reporting (k=5 is primary for Tables VII/VIII per ADR 0001)
    for c in range(6):
        k = 5
        top_k = top_neurons[c][:k]
        ablated_accs = ablate_and_evaluate(model, acts_1152, labels, top_k, dataset_mean_acts)
        
        # Accuracy drops (positive means accuracy went down)
        acc_drops = baseline_accs - ablated_accs
        
        target_drop = acc_drops[c]
        non_target_drops = np.delete(acc_drops, c)
        mean_abs_non_target_drop = np.mean(np.abs(non_target_drops))
        
        # The ADR 0001 Selectivity threshold
        is_selective = (target_drop > 0) and (target_drop >= (2.5 * mean_abs_non_target_drop))
        
        results.append({
            'model': model_name,
            'class': EMOTION_CLASSES[c],
            'baseline_acc': round(baseline_accs[c]*100, 2),
            'target_drop': round(target_drop*100, 2),
            'mean_nt_drop': round(mean_abs_non_target_drop*100, 2),
            'ratio': round(target_drop / (mean_abs_non_target_drop + 1e-8), 2),
            'causally_selective': is_selective
        })
        
    df = pd.DataFrame(results)
    print(f"\n{model_name.upper()} MODEL - k=5 Causal Ablation Results:")
    print(df.to_string(index=False))
    
    # Save the sweep for Figure 6 and Phase D retention analysis
    # (Including k=1,3,5,10)
    sweep_results = {}
    for k_val in K_VALUES:
        sweep_results[k_val] = {}
        for c in range(6):
            top_k = top_neurons[c][:k_val]
            ablated_accs = ablate_and_evaluate(model, acts_1152, labels, top_k, dataset_mean_acts)
            sweep_results[k_val][EMOTION_CLASSES[c]] = (baseline_accs - ablated_accs).tolist()
            
    # Save JSON artifacts
    out_path = os.path.join(results_dir, f'phase_c_{model_name}_ablation_sweep.json')
    with open(out_path, 'w') as f:
        json.dump(sweep_results, f, indent=2)
        
    return df, sweep_results

# Execute sweeps
base_df, base_sweep = run_sweep(base_model, base_1152, base_acts_tier1, labels_tier1, base_top_neurons, 'base')
ft_df, ft_sweep = run_sweep(ft_model, ft_1152, ft_acts_tier1, labels_tier1, ft_top_neurons, 'finetuned')

print("\nPhase C Causal Ablation Complete. Sweep artifacts saved to the results directory.")

=== Phase C: Causal Ablation Sweep ===

BASE MODEL - k=5 Causal Ablation Results:
model     class  baseline_acc  target_drop  mean_nt_drop      ratio  causally_selective
 base     anger         75.00         0.00          0.83       0.00               False
 base   disgust         87.50         0.00          0.00       0.00               False
 base      fear         54.17         0.00          0.83       0.00               False
 base happiness         83.33         0.00          0.83       0.00               False
 base   sadness         70.83         0.00          0.83       0.00               False
 base  surprise         87.50         4.17          0.00 4166666.67                True

FINETUNED MODEL - k=5 Causal Ablation Results:
    model     class  baseline_acc  target_drop  mean_nt_drop  ratio  causally_selective
finetuned     anger         79.17          0.0          0.00    0.0               False
finetuned   disgust         83.33          0.0          0.00    0.0           